# EDA - Wine Quality Dataset

Análisis exploratorio del dataset WineQT, que recoge medidas físico-químicas de vinos tintos y una puntuación de calidad asignada por catadores. El objetivo final es predecir la variable `quality` mediante regresión.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
df = pd.read_csv('WineQT.csv', sep=';')
df.shape

In [ ]:
df.head()

In [ ]:
df.columns.tolist()

La columna Id es un índice artificial sin valor predictivo, la eliminamos.

In [ ]:
df = df.drop(columns=['Id'])
df.shape

## Tipos de datos y estadísticas básicas

In [ ]:
df.dtypes

In [ ]:
df.describe()

## Valores nulos y duplicados

In [ ]:
df.isnull().sum()

In [ ]:
print(f'Filas duplicadas: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'Tamaño tras eliminar duplicados: {df.shape}')

No hay valores nulos. Sí existen filas duplicadas las cuales eliminamos.

## Variable objetivo: quality

In [ ]:
df['quality'].value_counts().sort_index()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de quality')
axes[0].set_xlabel('quality')
axes[0].set_ylabel('frecuencia')
axes[0].tick_params(axis='x', rotation=0)

df['quality'].plot(kind='box', ax=axes[1])
axes[1].set_title('Boxplot de quality')

plt.tight_layout()
plt.show()

La variable `quality` toma valores entre 3 y 8. Las clases 5 y 6 concentran el 82% de los datos. El dataset está un poco desbalanceado. Las clases extremas (3 y 8) tienen muy pocos ejemplos, lo que puede hacer que el modelo tienda a predecir valores centrales.

## Distribución de las variables

In [ ]:
features = [c for c in df.columns if c != 'quality']

df[features].hist(bins=30, figsize=(14, 9), color='steelblue', edgecolor='white')
plt.suptitle('Distribución de las variables físico-químicas', y=1.01)
plt.tight_layout()
plt.show()

## Detección de outliers

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    flierprops=dict(marker='o', markerfacecolor='tomato', markersize=3))
    axes[i].set_title(col, fontsize=9)

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots — detección de outliers', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cuantificación de outliers por IQR
resumen = []
for col in features:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    resumen.append({'variable': col, 'n_outliers': n_out, '%': round(n_out/len(df)*100, 1)})

pd.DataFrame(resumen).set_index('variable')

Se detectan outliers especialmente en `chlorides`, `residual sugar` y `sulphates`. Al tratarse de mediciones físico-químicas reales (no errores), los mantenemos, su eliminación podria distorsionar el proyecto.

## Correlación

In [ ]:
corr = df.corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación')
plt.show()

In [ ]:
# Correlación con la variable objetivo
corr_quality = df.corr()['quality'].drop('quality').sort_values(key=abs, ascending=False)
corr_quality

`alcohol` es la variable con mayor correlación positiva con `quality` (~0.48) y `volatile acidity` la mayor negativa (~-0.40). También se aprecia multicolinealidad entre `free sulfur dioxide` y `total sulfur dioxide` (r~0.67), lo que puede afectar a los coeficientes de la regresión.

## Relación entre las features más relevantes y quality

In [ ]:
top_vars = corr_quality.abs().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, col in enumerate(top_vars):
    sns.boxplot(data=df, x='quality', y=col, ax=axes[i], palette='muted')
    axes[i].set_title(col)

plt.suptitle('Variables más correlacionadas con quality', y=1.01)
plt.tight_layout()
plt.show()

Los boxplots confirman las tendencias detectadas en la correlación. Se observa mucha varianza dentro de cada grupo de calidad, lo que indica que ninguna variable por sí sola es suficiente para predecir `quality`. El modelo necesitará combinar varias features.

In [ ]:
df.to_csv('WineQT_clean.csv', index=False)